In [4]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
import pandas as pd


In [5]:
train_set = pd.read_csv('/Users/althealam/Desktop/GitHub/Generative-Retrieval-for-Multi-Destination-Trips-via-RQ-VAE/data/train_set.csv')
test_set = pd.read_csv('/Users/althealam/Desktop/GitHub/Generative-Retrieval-for-Multi-Destination-Trips-via-RQ-VAE/data/test_set.csv')

In [6]:
# 1. 准备序列 (字符串格式)
sentences = train_set.groupby('utrip_id')['city_id'].apply(lambda x: [str(i) for i in x]).tolist()

# 2. 训练 Word2Vec (核心：让 ID 具备空间意义)
w2v = Word2Vec(sentences, vector_size=64, window=5, min_count=1, workers=4)

# 3. 提取所有城市的向量并进行残差量化 (RQ-VAE 核心逻辑)
all_cities = [str(c) for c in train_set['city_id'].unique()]
vectors = np.array([w2v.wv[c] for c in all_cities])

# 第一层聚类 (大类)
kmeans1 = KMeans(n_clusters=32, random_state=42, n_init=10).fit(vectors)
codes1 = kmeans1.labels_

# 第二层聚类 (残差小类)
residuals = vectors - kmeans1.cluster_centers_[codes1]
kmeans2 = KMeans(n_clusters=32, random_state=42, n_init=10).fit(residuals)
codes2 = kmeans2.labels_

# 建立映射表：City_ID -> (Code1, Code2)
city_to_codes = {int(all_cities[i]): (codes1[i], codes2[i]) for i in range(len(all_cities))}